In [1]:
%pip install optuna


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import KFold

ARTIFACTS = Path("../artifacts")
RESULTS = Path("../results")

df = pd.read_csv(ARTIFACTS / "clean_data.csv")
with open(ARTIFACTS / "split_config.json") as f:
    split_config = json.load(f)

SEED = split_config["random_state"]
TARGET = "CO2 uptake (mmol/g)"
FEATURES = [c for c in df.columns if c not in ["row_id", TARGET, "split"]]

train = df[df["split"] == "train"]
X_train, y_train = train[FEATURES], train[TARGET]

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

def cv_rmse(model):
    pipe = make_pipeline(SimpleImputer(strategy="median"), model)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error")
    return -scores.mean()

In [5]:
from sklearn.ensemble import RandomForestRegressor

default_rf = RandomForestRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
print(round(cv_rmse(default_rf), 3))

0.779


In [7]:
import optuna

def rf_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth":        trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":     trial.suggest_float("max_features", 0.3, 1.0),
    }
    model = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
    return cv_rmse(model)

In [8]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

rf_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
rf_study.optimize(rf_objective, n_trials=30, show_progress_bar=True)

print("Best CV RMSE:", round(rf_study.best_value, 3))
print("Best settings:", rf_study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Best CV RMSE: 0.737
Best settings: {'n_estimators': 200, 'max_depth': 18, 'min_samples_leaf': 1, 'max_features': 0.48470637700561614}


In [9]:
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [10]:
def build_model(name, trial):
    if name == "Random Forest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
            max_depth=trial.suggest_int("max_depth", 3, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_float("max_features", 0.3, 1.0),
            random_state=SEED, n_jobs=-1)

    if name == "Extra Trees":
        return ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
            max_depth=trial.suggest_int("max_depth", 3, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_float("max_features", 0.3, 1.0),
            random_state=SEED, n_jobs=-1)

    if name == "Gradient Boosting":
        return GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            random_state=SEED)

    if name == "XGBoost":
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            random_state=SEED, n_jobs=-1)

    if name == "LightGBM":
        return LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 8, 64),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 40),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            random_state=SEED, verbose=-1)

    if name == "CatBoost":
        return CatBoostRegressor(
            iterations=trial.suggest_int("iterations", 200, 1000, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            depth=trial.suggest_int("depth", 4, 8),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
            random_state=SEED, verbose=0)

In [11]:
MODEL_NAMES = ["Random Forest", "Extra Trees", "Gradient Boosting",
               "XGBoost", "LightGBM", "CatBoost"]
N_TRIALS = 40

studies = {}
for name in MODEL_NAMES:
    print(name)
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: cv_rmse(build_model(name, trial)),
                   n_trials=N_TRIALS, show_progress_bar=True)
    studies[name] = study
    print(f"  best CV RMSE: {study.best_value:.3f}\n")

Random Forest


  0%|          | 0/40 [00:00<?, ?it/s]

  best CV RMSE: 0.737

Extra Trees


  0%|          | 0/40 [00:00<?, ?it/s]

  best CV RMSE: 0.738

Gradient Boosting


...

  best CV RMSE: 0.727

LightGBM


  0%|          | 0/40 [00:00<?, ?it/s]

  best CV RMSE: 0.723

CatBoost


  0%|          | 0/40 [00:00<?, ?it/s]

  best CV RMSE: 0.689



In [12]:
before = pd.read_csv(RESULTS / "04_model_comparison.csv")[["model", "cv_rmse"]]
after = pd.DataFrame({
    "model": list(studies),
    "tuned_cv_rmse": [s.best_value for s in studies.values()],
})

comparison = before.merge(after, on="model")
comparison["improvement"] = comparison["cv_rmse"] - comparison["tuned_cv_rmse"]
comparison = comparison.sort_values("tuned_cv_rmse").round(3)
comparison

,model,cv_rmse,tuned_cv_rmse,improvement
0,CatBoost,0.703,0.689,0.014
4,LightGBM,0.779,0.723,0.056
1,XGBoost,0.758,0.727,0.031
5,Gradient Boosting,0.787,0.729,0.058
3,Random Forest,0.779,0.737,0.042
2,Extra Trees,0.765,0.738,0.027


In [13]:
best_params = {name: s.best_params for name, s in studies.items()}

with open(ARTIFACTS / "best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

comparison.to_csv(RESULTS / "05_tuning_comparison.csv", index=False)
print("saved")

saved
